In [4]:
import time
import torch
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset
from models.gpt2 import build_gpt2
from models.loss import gpt2_loss


def prepare_dataset(tokenizer, split="train", seq_len=256):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1")[split]

    def encode(ex):
        tok = tokenizer(
            ex["text"],
            truncation=True,
            padding="max_length",
            max_length=seq_len,
        )
        return {
            "input_ids": tok["input_ids"],
            "attention_mask": tok["attention_mask"],
        }

    ds = ds.map(encode, batched=True, remove_columns=["text"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
    return ds


def train_single_gpu(num_epochs=1, bs=16, lr=2e-5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = build_gpt2().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    loader = DataLoader(
        prepare_dataset(tokenizer, "train"),
        batch_size=bs,
        shuffle=True,
        num_workers=4,
        pin_memory=(device.type == "cuda"),
    )

    total_samples = 0
    peak_mem_gb = 0.0

    for epoch in range(num_epochs):
        model.train()
        epoch_start = time.perf_counter()
        window_start = time.perf_counter()
        window_samples = 0

        for i, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attn = batch["attention_mask"].to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                outputs = model(input_ids, attention_mask=attn, labels=input_ids)
                loss = gpt2_loss(outputs)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

            window_samples += bs
            total_samples += bs

            if i % 20 == 0 and i > 0:
                if device.type == "cuda":
                    torch.cuda.synchronize()
                elapsed = time.perf_counter() - window_start
                throughput = window_samples / elapsed

                if device.type == "cuda":
                    mem_gb = torch.cuda.max_memory_allocated() / 1024**3
                    peak_mem_gb = max(peak_mem_gb, mem_gb)
                    mem_str = f"mem={mem_gb:.2f}GB"
                else:
                    mem_str = ""

                print(
                    f"epoch={epoch} step={i:>5d} loss={loss.item():.4f} "
                    f"throughput={throughput:.1f} samples/s {mem_str}"
                )

                window_start = time.perf_counter()
                window_samples = 0

        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_time = time.perf_counter() - epoch_start
        avg_throughput = total_samples / epoch_time

        print(
            f"\n=== epoch {epoch} done | "
            f"time={epoch_time:.1f}s | "
            f"avg_throughput={avg_throughput:.1f} samples/s | "
            f"peak_mem={peak_mem_gb:.2f}GB ===\n"
        )

In [ ]:
# cell 1: your code definitions, already present

# cell 2: call
train_single_gpu(num_epochs=1, bs=4, lr=2e-5)

cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


epoch=0 step=0 loss=10.9604 throughput=60.37 samples/s
epoch=0 step=20 loss=6.5388 throughput=15.28 samples/s
epoch=0 step=40 loss=1.7916 throughput=15.18 samples/s
epoch=0 step=60 loss=2.5475 throughput=15.19 samples/s
epoch=0 step=80 loss=1.3191 throughput=15.02 samples/s
epoch=0 step=100 loss=0.4417 throughput=14.98 samples/s
epoch=0 step=120 loss=0.9262 throughput=14.85 samples/s
epoch=0 step=140 loss=2.4628 throughput=14.73 samples/s
epoch=0 step=160 loss=2.1370 throughput=14.68 samples/s
epoch=0 step=180 loss=4.2151 throughput=14.64 samples/s
epoch=0 step=200 loss=3.5811 throughput=14.56 samples/s
epoch=0 step=220 loss=3.2918 throughput=14.52 samples/s
epoch=0 step=240 loss=1.0443 throughput=14.41 samples/s
epoch=0 step=260 loss=0.0516 throughput=14.36 samples/s
epoch=0 step=280 loss=5.0398 throughput=14.27 samples/s
epoch=0 step=300 loss=3.8310 throughput=14.19 samples/s
epoch=0 step=320 loss=3.4813 throughput=14.16 samples/s
epoch=0 step=340 loss=0.9570 throughput=14.03 samples

In [ ]:
!pwd
!ls /content
%cd /content
!git clone https://github.com/krishnajha23/training.git
%cd /content/training

/content/training
sample_data  training
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: destination path 'training' already exists and is not an empty directory.
/content/training


In [1]:
!nvidia-smi

Sat Mar 21 16:27:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----